# Phase 2 — unchanged Qwen3-4B on DEV (PREREGISTRATION §5 with Amendment 1)
Run this **once, before any training run**. 40 DEV families × E/R/C = 120 generations with the unchanged model.
Frozen protocol (PREREGISTRATION §4): prompt `test1-english-final-v2`, greedy non-thinking decoding, NF4 + FP16, SDPA math, output cap 1,024 tokens, scorer v1.0.0, engine seed 11. **Do not change any setting.**
The notebook embeds `phase2/IMPLEMENTATION_MANIFEST.md` (SHA-256 `18ab0cb5f7ca592e36ecb434b8cd6c7d6e0194d4c412e49912d9bd5e3c813f16`) and refuses to run if any embedded code or data differs from it, or if the pinned tokenizer/config files differ (Amendment 1 item 14). It refuses any stack other than the exact §2 stack (torch 2.11.0+cu130 included) on a T4 (Amendment 2 item 3).
At the end it prints **high unchanged-DEV accuracy (ceiling-risk proxy)** with k/40 and its Wilson 95% interval; the flag fires at C ≥ 38/40 (Amendment 1 item 7). It is a proxy only: Δ_C stays first in the report whatever it shows. DEV is a sanity set: it never selects anything. No FINAL_TEST data is in this notebook.
**Baseline first (Amendment 2 item 6).** The complete run is recorded on Drive (`MyDrive/Hinglish_Phase2/baseline_dev/base_baseline_record.json`: results-ZIP SHA-256 and implementation-manifest SHA-256). The Train notebook refuses the first training attempt without it, and every attempt and the adapter freeze record its ZIP hash. Do not delete or rerun it.
**Stale lock (Amendment 2 item 5).** If a runtime was killed during an evaluation, re-running may stop with "Run lock exists". First make sure no other Colab session is running this notebook, then set `RECOVER_STALE_LOCK = True` and re-run all: the lock is removed, a timestamped record is written next to the run, and generation resumes from the stored journal (nothing is regenerated). Set it back to False afterwards.
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (about 15–20 minutes). Drive is on by default so a disconnect resumes (re-run all; finished prompts are kept, never regenerated). A results ZIP with checksums downloads at the end; send it unchanged.


In [ ]:
import os, sys, json, base64, io, zipfile, hashlib, subprocess, time
from pathlib import Path
USE_DRIVE = True  # resume after a disconnect
RECOVER_STALE_LOCK = False  # Amendment 2 item 5: set True ONLY after confirming that no other Colab session is running this evaluation (a killed runtime left RUNNING.lock); the recovery is logged with a timestamp and stored outputs are never regenerated
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'  # download bars would become thousands of streamed lines
PACKAGE = Path('/content/hinglish_phase2') if Path('/content').exists() else Path.cwd() / 'hinglish_phase2'
PACKAGE.mkdir(parents=True, exist_ok=True)
blob = base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest() == '2a9993d9510497b123e3779028f15a686430848d4864ab143500f74abbb73886'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(PACKAGE)
INPUTS = PACKAGE / 'inputs'
# Implementation manifest (PREREGISTRATION Amendment 1 item 14): refuse to run if the embedded code differs from it.
MANIFEST_TEXT = base64.b64decode('...').decode('utf-8')
MANIFEST_SHA256 = '18ab0cb5f7ca592e36ecb434b8cd6c7d6e0194d4c412e49912d9bd5e3c813f16'
assert hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest() == MANIFEST_SHA256, 'Embedded IMPLEMENTATION_MANIFEST.md was changed: refusing to run'
MANIFEST = json.loads(MANIFEST_TEXT.split('```json\n', 1)[1].split('\n```', 1)[0])
assert MANIFEST['preregistration']['sha256'] == 'a22c81fb33b4d3339770a2b28c51d8d3706af2b5dd038d46aa89fbb56139714d', 'Manifest is for another PREREGISTRATION.md: refusing to run'
assert MANIFEST['notebook_payloads']['Phase2_Baseline_DEV.ipynb'] == hashlib.sha256(blob).hexdigest(), 'Notebook payload differs from the implementation manifest: refusing to run'
for member, source in {'hinglish_pilot/__init__.py': 'test1_pilot/hinglish_pilot/__init__.py', 'hinglish_pilot/data.py': 'test1_pilot/hinglish_pilot/data.py', 'hinglish_pilot/engine.py': 'test1_pilot/hinglish_pilot/engine.py', 'hinglish_pilot/hf_backend.py': 'test1_pilot/hinglish_pilot/hf_backend.py', 'hinglish_pilot/report.py': 'test1_pilot/hinglish_pilot/report.py', 'hinglish_pilot/scorer_v1.py': 'test1_pilot/hinglish_pilot/scorer_v1.py', 'secondary_analysis/__init__.py': 'secondary_analysis/__init__.py', 'secondary_analysis/cmi.py': 'secondary_analysis/cmi.py', 'secondary_analysis/numeric_endpoint.py': 'secondary_analysis/numeric_endpoint.py', 'secondary_analysis/report_secondary.py': 'secondary_analysis/report_secondary.py', 't4_smoke/train_smoke.py': 't4_smoke/train_smoke.py', 't4_smoke/labels.py': 't4_smoke/labels.py', 't4_smoke/protocol.py': 't4_smoke/protocol.py', 't4_smoke/train_phase2.py': 't4_smoke/train_phase2.py', 't4_smoke/phase2_eval.py': 't4_smoke/phase2_eval.py', 'inputs/dev.v2.json': 'phase2/dev.v2.json'}.items():
    assert hashlib.sha256((PACKAGE / member).read_bytes()).hexdigest() == MANIFEST['files'][source], f'{member} differs from the implementation manifest: refusing to run'
(INPUTS / 'IMPLEMENTATION_MANIFEST.md').write_text(MANIFEST_TEXT, encoding='utf-8')
print('Implementation manifest verified:', MANIFEST_SHA256)
if USE_DRIVE and Path('/content').exists():  # Drive only inside Colab
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = Path('/content/drive/MyDrive/Hinglish_Phase2')
else:
    STORE = PACKAGE / 'runs'
STORE.mkdir(parents=True, exist_ok=True)
try:  # Qwen3-4B is ungated; a token only avoids anonymous rate limits.
    from google.colab import userdata
    os.environ['HF_TOKEN'] = (userdata.get('HF_TOKEN') or '').strip() or os.environ.get('HF_TOKEN', '')
except Exception as error:
    print('No HF_TOKEN Colab secret (' + type(error).__name__ + '); downloading anonymously.')
if not os.environ.get('HF_TOKEN'):
    os.environ.pop('HF_TOKEN', None)

def step(log_dir, script, *args):
    """Run one phase in a fresh Python process, stream its output and log the attempt."""
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', str(PACKAGE / 't4_smoke' / script), *map(str, args)]
    started, returncode = time.time(), None
    try:
        with open(log_dir / 'log.txt', 'a') as log, subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                                     text=True, errors='replace') as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
            except BaseException:  # interrupt: never leave a phase running on the GPU behind the notebook
                process.kill()
                process.wait()
                log.write('[notebook] interrupted; phase process killed\n')
                raise
        returncode = process.returncode
    finally:
        with open(log_dir / 'attempts.jsonl', 'a') as attempts:
            attempts.write(json.dumps({'script': script, 'args': list(map(str, args)), 'returncode': returncode,
                                       'started': started, 'seconds': time.time() - started}) + '\n')
    return returncode

def zip_results(folder, archive, prefix):
    """All small files under ``folder``; weights, optimizer state and per-attempt journals stay out (hashes listed)."""
    files, large = [], {}
    for path in sorted(Path(folder).rglob('*')):
        if not path.is_file() or 'journal' in path.relative_to(folder).parts or path.name.endswith('.lock'):
            continue
        if path.suffix in ('.pt', '.safetensors'):
            large[str(path.relative_to(folder))] = hashlib.sha256(path.read_bytes()).hexdigest()
        else:
            files.append(path)
    sums = ''.join(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(folder)}\n' for p in files)
    sums += ''.join(f'{digest}  {name}  (not included)\n' for name, digest in large.items())
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in files:
            z.write(path, str(Path(prefix) / path.relative_to(folder)))
        z.writestr(str(Path(prefix) / 'SHA256SUMS.txt'), sums)
    print(sums)
    print('Return:', archive, 'SHA-256', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
    try:
        from google.colab import files as colab_files
        colab_files.download(str(archive))
    except ImportError:
        print('Local: ZIP saved on disk.')


In [ ]:
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:  # CPU runtime: no nvidia-smi at all
    gpu = ''
print('GPU:', gpu or 'none')
assert 'T4' in gpu, 'Select Runtime → Change runtime type → T4 GPU'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements-phase2.txt')], check=True)
# A fresh interpreter sees the installed versions; this kernel never imports torch.
STACK = json.loads(subprocess.run([sys.executable, '-c', 'import sys; sys.path.insert(0, sys.argv[1]); import train_smoke as t, json; '
                                   'v = t.versions(); print(json.dumps({"versions": v, "mismatches": t.stack_mismatches(v)}))',
                                   str(PACKAGE / 't4_smoke')], capture_output=True, text=True, check=True).stdout)
print(json.dumps(STACK['versions'], indent=1))
STACK_MISMATCHES = STACK['mismatches']
print('D021 stack mismatches:', STACK_MISMATCHES or 'none')
# Amendment 2 item 3: only the exact §2 stack on a T4 can produce a study run; there is no override.
assert not STACK_MISMATCHES, 'D021 stack mismatch: no study run until a dated amendment (Amendment 2 item 3): ' + '; '.join(STACK_MISMATCHES)
# Tokenizer, chat template and config files at the pinned revision must match the implementation manifest.
HUB_CHECK = """
import hashlib, json, os, sys
from huggingface_hub import hf_hub_download
model = json.loads(sys.argv[1])
for name, expected in sorted(model['hub_files'].items()):
    path = hf_hub_download(model['id'], name, revision=model['revision'], token=os.environ.get('HF_TOKEN') or None)
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert actual == expected, f'{name} at the pinned revision has SHA-256 {actual}, manifest {expected}: refusing to run'
print('Pinned Hub files match the implementation manifest:', ', '.join(sorted(model['hub_files'])))
"""
print(subprocess.run([sys.executable, '-c', HUB_CHECK, json.dumps(MANIFEST['model'])], capture_output=True, text=True, check=True).stdout)


In [ ]:
RUN_DIR = STORE / 'baseline_dev' / 'base'
step(RUN_DIR.parent, 'phase2_eval.py', 'evaluate', '--split', 'dev', '--inputs', INPUTS, '--run-dir', RUN_DIR,
     '--implementation-manifest', INPUTS / 'IMPLEMENTATION_MANIFEST.md', *(['--recover-stale-lock'] if RECOVER_STALE_LOCK else []))
FLAG = RUN_DIR.parent / 'base_ceiling_flag.json'
RECORD = RUN_DIR.parent / 'base_baseline_record.json'
assert FLAG.exists() and RECORD.exists(), 'Evaluation incomplete (see above): re-run all to resume, then download'
flag = json.loads(FLAG.read_text())
low, high = flag['wilson95']
print(f"\n{flag['name']}: {'FIRES' if flag['flag'] else 'does not fire'} — unchanged DEV C {flag['k']}/{flag['n']}"
      f" (Wilson 95% {100 * low:.1f}–{100 * high:.1f}%; threshold >= 38/40)")
record = json.loads(RECORD.read_text())
assert record['implementation_manifest_sha256'] == MANIFEST_SHA256, 'Baseline was recorded under another implementation manifest'
print('Unchanged-DEV baseline recorded for training (Amendment 2 item 6):', record['results_zip'], record['results_zip_sha256'])


## Download
If the run stopped early, re-run all cells first. Then run this cell.


In [ ]:
zip_results(STORE / 'baseline_dev', PACKAGE / 'Phase2_Baseline_DEV_results.zip', 'baseline_dev')
